# 14. 다중 독성 endpoint 확장 (hERG 등)

## 이번 노트북에서 할 것
- TDC에서 hERG(심장 이온채널 저해) 데이터셋 로드, ECFP+RandomForest로 baseline 학습
- (여유 시) DILI, Skin_Reaction 등 추가 endpoint도 같은 패턴으로 확장
- 새 endpoint를 iterative_fix_loop의 재평가 단계에 통합
- LLM 기반 vs 규칙 기반(candidate_idx=0)의 치환이 새 endpoint에서 얼마나
  차이나는지 비교 (12번에서 Tox21/Ames로 했던 것과 같은 패턴)
- 3개 endpoint(Tox21, Ames, hERG) 종합 비교표 작성

## 간략한 정리 (13까지)
- 라이브러리 11개 규칙 (문헌 7 + MMPA데이터 2 + Sulfonic_acid_2/imine_1 2,
  다만 이름 불일치/이온형 누락 버그 있었고 수정 완료)
- held-out 커버리지 166 -> 227개 (+37%)
- 3단계 검증(구조규칙->Tox21->Ames) 완성, 167개 표본 둘 다 유의(p<0.0001 수준)
- 문헌형/데이터형 분류 기준 확정, 다음 확장 후보 6개 분류 완료

## 다음에 해야 할 것 (오늘 끝나면)
- 학생이 문헌형 규칙(catechol, thiol_2 등) 직접 추가
- 제안서에 다중 endpoint 검증 결과 반영
- 제안서 hwpx 작업 착수

In [1]:
# 셀 1
!pip install rdkit -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install fuzzywuzzy python-Levenshtein -q

In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

fatal: destination path 'laidd-2026' already exists and is not an empty directory.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import joblib

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

data = load_tox21_clean()

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return _generator.GetFingerprintAsNumPy(mol) if mol else None

print("도구 로드 확인 완료")

[01:29:23] WARNING: not removing hydrogen atom without neighbors
[01:29:24] Explicit valence for atom # 8 Al, 6, is greater than permitted
[01:29:24] Explicit valence for atom # 3 Al, 6, is greater than permitted
[01:29:24] Explicit valence for atom # 4 Al, 6, is greater than permitted
[01:29:24] Explicit valence for atom # 4 Al, 6, is greater than permitted
[01:29:25] Explicit valence for atom # 9 Al, 6, is greater than permitted
[01:29:25] Explicit valence for atom # 5 Al, 6, is greater than permitted
[01:29:25] Explicit valence for atom # 16 Al, 6, is greater than permitted
[01:29:25] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[01:29:26] WARNING: not removing hydrogen atom without neighbors


도구 로드 확인 완료


In [5]:
# Tox21 baseline
from tdc.single_pred import Tox
ames_data = Tox(name='AMES')
ames_split = ames_data.get_split()

X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
task_cols = data['task_cols']
classifiers = {}
for i, task in enumerate(task_cols):
    train_mask = w_train[:, i] == 1
    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train[train_mask], y_train[train_mask, i])
    classifiers[task] = clf
print("Tox21 baseline 재학습 완료")

# Ames
from tdc.single_pred import Tox
ames_data = Tox(name='AMES')
ames_split = ames_data.get_split()

def prepare_split_generic(df):
    df = df.copy()
    df['mol_valid'] = df['Drug'].apply(lambda s: Chem.MolFromSmiles(s) is not None)
    df_clean = df[df['mol_valid']].reset_index(drop=True)
    X = np.stack(df_clean['Drug'].apply(smiles_to_ecfp).values)
    y = df_clean['Y'].values
    return X, y, df_clean['Drug'].values

X_train_ames, y_train_ames, _ = prepare_split_generic(ames_split['train'])
ames_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
ames_clf.fit(X_train_ames, y_train_ames)
print("Ames baseline 재학습 완료")

def predict_ames(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = smiles_to_ecfp(smiles).reshape(1, -1)
    return ames_clf.predict_proba(fp)[0][1]

def predict_tox21_avg(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = smiles_to_ecfp(smiles).reshape(1, -1)
    return np.mean([classifiers[t].predict_proba(fp)[0][1] for t in task_cols])

Found local copy...
Loading...
Done!
Found local copy...
Loading...
Done!


Tox21 baseline 재학습 완료
Ames baseline 재학습 완료


In [6]:
herg_data = Tox(name='hERG')
herg_split = herg_data.get_split()
print("Train:", herg_split['train'].shape, "Valid:", herg_split['valid'].shape, "Test:", herg_split['test'].shape)

X_train_herg, y_train_herg, _ = prepare_split_generic(herg_split['train'])
X_valid_herg, y_valid_herg, _ = prepare_split_generic(herg_split['valid'])
X_test_herg, y_test_herg, _ = prepare_split_generic(herg_split['test'])

herg_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
herg_clf.fit(X_train_herg, y_train_herg)

herg_valid_auc = roc_auc_score(y_valid_herg, herg_clf.predict_proba(X_valid_herg)[:,1])
herg_test_auc = roc_auc_score(y_test_herg, herg_clf.predict_proba(X_test_herg)[:,1])
print(f"Valid AUROC: {herg_valid_auc:.3f}, Test AUROC: {herg_test_auc:.3f}")

def predict_herg(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = smiles_to_ecfp(smiles).reshape(1, -1)
    return herg_clf.predict_proba(fp)[0][1]

Found local copy...
Loading...
Done!


Train: (458, 3) Valid: (66, 3) Test: (131, 3)


[01:30:27] WARNING: not removing hydrogen atom without neighbors
[01:30:27] WARNING: not removing hydrogen atom without neighbors
[01:30:27] WARNING: not removing hydrogen atom without neighbors
[01:30:27] WARNING: not removing hydrogen atom without neighbors


Valid AUROC: 0.836, Test AUROC: 0.846


In [7]:
import joblib, json
from datetime import datetime

joblib.dump(herg_clf, 'models/tox21_classifier/herg_rf.pkl')
with open('models/tox21_classifier/herg_rf_meta.json', 'w') as f:
    json.dump({
        "model_type": "RandomForestClassifier",
        "dataset": "TDC hERG (Karim et al. 2021, via PyTDC --no-deps)",
        "train_size": 458, "valid_size": 66, "test_size": 131,
        "valid_auc": 0.836, "test_auc": 0.846,
        "created_at": datetime.now().isoformat(),
    }, f, indent=2)
print("저장 완료")

저장 완료


In [ ]:
# 커밋 - 이후 다시 실행하지말 것
!git add models/tox21_classifier/herg_rf_meta.json
!git commit -m "Add hERG cardiotoxicity model (AUROC 0.846, via PyTDC --no-deps to avoid numpy/scipy version conflicts)"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 346f3ad] Add hERG cardiotoxicity model (AUROC 0.846, via PyTDC --no-deps to avoid numpy/scipy version conflicts)
 1 file changed, 10 insertions(+)
 create mode 100644 models/tox21_classifier/herg_rf_meta.json
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 722 bytes | 722.00 KiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Dec32th/laidd-2026.git
   7b2400e..346f3ad  main -> main


In [8]:
def predict_herg(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = smiles_to_ecfp(smiles).reshape(1, -1)
    return herg_clf.predict_proba(fp)[0][1]

In [9]:
!pip install openai -q

In [10]:
from openai import OpenAI

dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(
    api_key=dashscope_key,
    base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
)
print("Qwen 클라이언트 준비 완료")

Qwen 클라이언트 준비 완료


In [11]:
target_rules_all = ["alkyl_halide", "nitro_group", "aniline", "acyl_halide", "aldehyde",
                     "Sulfonic_acid_2", "imine_1"]

verification_cases_v3 = []
for s in data['smiles_test']:
    problems = detect_toxicophores(s)
    known = [p for p in problems if p['rule_name'] in target_rules_all]
    if not known:
        continue
    rule = known[0]['rule_name']
    verification_cases_v3.append({"original": s, "rule": rule})
    if len(verification_cases_v3) >= 20:
        break

print(f"비교 대상: {len(verification_cases_v3)}개")

비교 대상: 20개


In [12]:
comparison_results = []
for c in verification_cases_v3:
    rule_fixed = propose_fix(c['original'], c['rule'], candidate_idx=0)
    if rule_fixed is None or not rule_fixed['is_valid']:
        continue

    llm_candidate = ask_llm_which_candidate_to_use(client_qwen, "qwen3.8-max-preview", c['original'], c['rule'], client_type="openai_compatible")
    llm_fixed = propose_fix(c['original'], c['rule'], llm_candidate['candidate_idx'])
    if llm_fixed is None or not llm_fixed['is_valid']:
        continue

    row = {"rule": c['rule'], "original": c['original']}
    for name, smi in [("rule", rule_fixed['new_smiles']), ("llm", llm_fixed['new_smiles'])]:
        row[f"{name}_tox21"] = predict_tox21_avg(smi)
        row[f"{name}_ames"] = predict_ames(smi)
        row[f"{name}_herg"] = predict_herg(smi)
    comparison_results.append(row)

print(f"완료된 비교: {len(comparison_results)}개")

완료된 비교: 15개


In [13]:
import pandas as pd

df_compare = pd.DataFrame(comparison_results)

# comparison_results에 포함된 original 분자들만 기준으로 원본 점수 계산
orig_tox21 = [predict_tox21_avg(r['original']) for r in comparison_results]
orig_ames = [predict_ames(r['original']) for r in comparison_results]
orig_herg = [predict_herg(r['original']) for r in comparison_results]

print(f"n = {len(df_compare)}\n")

print("=== 규칙기반 개선폭 (원본 대비, 음수=개선) ===")
print(f"Tox21: {np.mean(df_compare['rule_tox21']) - np.mean(orig_tox21):+.4f}")
print(f"Ames:  {np.mean(df_compare['rule_ames']) - np.mean(orig_ames):+.4f}")
print(f"hERG:  {np.mean(df_compare['rule_herg']) - np.mean(orig_herg):+.4f}")

print("\n=== LLM기반 개선폭 (원본 대비, 음수=개선) ===")
print(f"Tox21: {np.mean(df_compare['llm_tox21']) - np.mean(orig_tox21):+.4f}")
print(f"Ames:  {np.mean(df_compare['llm_ames']) - np.mean(orig_ames):+.4f}")
print(f"hERG:  {np.mean(df_compare['llm_herg']) - np.mean(orig_herg):+.4f}")

print("\n=== 규칙기반 vs LLM기반 직접 비교 (LLM이 더 나은 비율) ===")
for endpoint in ['tox21', 'ames', 'herg']:
    llm_better = sum(1 for i in range(len(df_compare)) if df_compare[f'llm_{endpoint}'][i] < df_compare[f'rule_{endpoint}'][i])
    print(f"{endpoint}: LLM이 더 낮은(더 안전한) 경우 {llm_better}/{len(df_compare)}개")

n = 15

=== 규칙기반 개선폭 (원본 대비, 음수=개선) ===
Tox21: -0.0187
Ames:  -0.1051
hERG:  -0.0335

=== LLM기반 개선폭 (원본 대비, 음수=개선) ===
Tox21: -0.0199
Ames:  -0.0998
hERG:  -0.0335

=== 규칙기반 vs LLM기반 직접 비교 (LLM이 더 나은 비율) ===
tox21: LLM이 더 낮은(더 안전한) 경우 1/15개
ames: LLM이 더 낮은(더 안전한) 경우 1/15개
herg: LLM이 더 낮은(더 안전한) 경우 2/15개


In [14]:
for rule in target_rules_all:
    info = get_replacement_candidates(rule)
    print(f"{rule}: 후보 {len(info['candidates'])}개 - {[c['name'] for c in info['candidates']]}")

alkyl_halide: 후보 2개 - ['hydroxyl (alcohol)', 'fluorine']
nitro_group: 후보 3개 - ['primary amine', 'sulfonamide', 'nitrile']
aniline: 후보 2개 - ['acetamide (acylated amine)', 'fluorine']
acyl_halide: 후보 2개 - ['amide', 'ester']
aldehyde: 후보 2개 - ['amide', 'alcohol']
Sulfonic_acid_2: 후보 2개 - ['sulfonamide', 'carboxylic acid']
imine_1: 후보 2개 - ['amine (reduced)', 'nitrile']


In [15]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "michael_acceptor": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "thiourea": {
        "problem_smarts": "NC(=S)N",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "황 원자를 산소로 대체, 유사한 형태를 유지하면서 반응성/대사 우려 감소"},
        ],
    },
    "acyl_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "phenol": {
        "problem_smarts": "[OX2H]",
        "candidates": [
            {"smiles": "Cl", "name": "chlorine",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->염소 치환 시 NR-ER, "
                          "NR-ER-LBD, SR-ARE 3개 assay 동시 개선 관찰됨. 페놀의 산화적 대사 "
                          "(quinone 형성 등) 경로를 차단하는 것으로 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
            {"smiles": "C", "name": "methyl",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 페놀->메틸 치환 시 NR-AR "
                          "assay 개선 관찰됨. 히드록실기 제거로 산화 취약성 감소 추정",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
    "amide": {
        "problem_smarts": "[NX3H1][CX3](=O)[#6]",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "MMPA 데이터 기반 발견: Tox21에서 N-메틸아마이드->우레아 치환 시 "
                          "SR-ARE assay 개선 관찰됨. 기존 thiourea->urea 치환과 같은 "
                          "계열(우레아 활용)로 일관성 있음",
             "source": "data-driven (Tox21 MMP analysis, charge-matched, Δmw<50)"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1": {
        "problem_smarts": "C=N[OX2H1]",
        "candidates": [
            {"smiles": "CN", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "옥심의 탈수 반응으로 니트릴을 얻는 것은 잘 알려진 화학 변환, "
                          "극성을 낮추면서 대사 불안정성 개선 (검증 필요)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [16]:
import importlib
import src.tools.replacement_library
import src.tools.molecule_editor

importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.molecule_editor)

from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix

for rule in target_rules_all:
    info = get_replacement_candidates(rule)
    print(f"{rule}: 후보 {len(info['candidates'])}개")

print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print(propose_fix("CNC(=O)c1ccccc1", "amide", candidate_idx=0))

alkyl_halide: 후보 2개
nitro_group: 후보 3개
aniline: 후보 2개
acyl_halide: 후보 2개
aldehyde: 후보 2개
Sulfonic_acid_2: 후보 2개
imine_1: 후보 2개
{'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
{'new_smiles': 'NC(=O)Nc1ccccc1', 'candidate_used': 'urea', 'rationale': 'MMPA 데이터 기반 발견: Tox21에서 N-메틸아마이드->우레아 치환 시 SR-ARE assay 개선 관찰됨. 기존 thiourea->urea 치환과 같은 계열(우레아 활용)로 일관성 있음', 'is_valid': True}


In [ ]:
!git add src/tools/replacement_library.py
!git commit -m "Add second candidate to acyl_halide, Sulfonic_acid_2, imine_1 to enable meaningful LLM candidate selection comparison"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 3a29e50] Add second candidate to acyl_halide, Sulfonic_acid_2, imine_1 to enable meaningful LLM candidate selection comparison
 1 file changed, 8 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 849 bytes | 849.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   346f3ad..3a29e50  main -> main


In [17]:
verification_cases_v4 = []
for s in data['smiles_test']:
    problems = detect_toxicophores(s)
    known = [p for p in problems if p['rule_name'] in target_rules_all]
    if not known:
        continue
    rule = known[0]['rule_name']
    verification_cases_v4.append({"original": s, "rule": rule})
    if len(verification_cases_v4) >= 30:
        break

print(f"비교 대상: {len(verification_cases_v4)}개")

비교 대상: 30개


In [18]:
comparison_results_v2 = []
for c in verification_cases_v4:
    rule_fixed = propose_fix(c['original'], c['rule'], candidate_idx=0)
    if rule_fixed is None or not rule_fixed['is_valid']:
        continue

    llm_candidate = ask_llm_which_candidate_to_use(client_qwen, "qwen3.8-max-preview", c['original'], c['rule'], client_type="openai_compatible")
    llm_fixed = propose_fix(c['original'], c['rule'], llm_candidate['candidate_idx'])
    if llm_fixed is None or not llm_fixed['is_valid']:
        continue

    row = {"rule": c['rule'], "original": c['original'],
           "llm_chose_idx": llm_candidate['candidate_idx'],
           "llm_reason": llm_candidate.get('reason', '')}
    for name, smi in [("rule", rule_fixed['new_smiles']), ("llm", llm_fixed['new_smiles'])]:
        row[f"{name}_tox21"] = predict_tox21_avg(smi)
        row[f"{name}_ames"] = predict_ames(smi)
        row[f"{name}_herg"] = predict_herg(smi)
    comparison_results_v2.append(row)

print(f"완료된 비교: {len(comparison_results_v2)}개")

# LLM이 0번(규칙기반 기본값)과 다른 걸 고른 비율
different_choice = sum(1 for r in comparison_results_v2 if r['llm_chose_idx'] != 0)
print(f"LLM이 기본값(0번)과 다른 후보 선택: {different_choice}/{len(comparison_results_v2)}개")

완료된 비교: 23개
LLM이 기본값(0번)과 다른 후보 선택: 7/23개


In [19]:
df_compare_v2 = pd.DataFrame(comparison_results_v2)

orig_tox21_v2 = [predict_tox21_avg(r['original']) for r in comparison_results_v2]
orig_ames_v2 = [predict_ames(r['original']) for r in comparison_results_v2]
orig_herg_v2 = [predict_herg(r['original']) for r in comparison_results_v2]

print(f"n = {len(df_compare_v2)}\n")

print("=== 규칙기반 개선폭 ===")
print(f"Tox21: {np.mean(df_compare_v2['rule_tox21']) - np.mean(orig_tox21_v2):+.4f}")
print(f"Ames:  {np.mean(df_compare_v2['rule_ames']) - np.mean(orig_ames_v2):+.4f}")
print(f"hERG:  {np.mean(df_compare_v2['rule_herg']) - np.mean(orig_herg_v2):+.4f}")

print("\n=== LLM기반 개선폭 ===")
print(f"Tox21: {np.mean(df_compare_v2['llm_tox21']) - np.mean(orig_tox21_v2):+.4f}")
print(f"Ames:  {np.mean(df_compare_v2['llm_ames']) - np.mean(orig_ames_v2):+.4f}")
print(f"hERG:  {np.mean(df_compare_v2['llm_herg']) - np.mean(orig_herg_v2):+.4f}")

# 다른 선택을 한 7개만 따로 떼서 비교 (여기가 진짜 궁금한 지점)
diff_only = df_compare_v2[df_compare_v2['llm_chose_idx'] != 0]
print(f"\n=== 다른 후보를 고른 {len(diff_only)}개만 비교 ===")
for endpoint in ['tox21', 'ames', 'herg']:
    llm_better = sum(1 for i in diff_only.index if diff_only[f'llm_{endpoint}'][i] < diff_only[f'rule_{endpoint}'][i])
    print(f"{endpoint}: LLM 선택이 더 안전한 경우 {llm_better}/{len(diff_only)}개")
    print(f"  규칙기반 평균: {diff_only[f'rule_{endpoint}'].mean():.3f}, LLM 평균: {diff_only[f'llm_{endpoint}'].mean():.3f}")

n = 23

=== 규칙기반 개선폭 ===
Tox21: -0.0084
Ames:  -0.1003
hERG:  -0.0278

=== LLM기반 개선폭 ===
Tox21: -0.0167
Ames:  -0.1264
hERG:  -0.0272

=== 다른 후보를 고른 7개만 비교 ===
tox21: LLM 선택이 더 안전한 경우 2/7개
  규칙기반 평균: 0.126, LLM 평균: 0.098
ames: LLM 선택이 더 안전한 경우 5/7개
  규칙기반 평균: 0.433, LLM 평균: 0.347
herg: LLM 선택이 더 안전한 경우 3/7개
  규칙기반 평균: 0.414, LLM 평균: 0.416


In [20]:
!git add -A
!git commit -m "Expand candidate diversity (30 percent different LLM choices); Ames shows clearest LLM advantage (5/7), Tox21/hERG mixed - endpoint-specific pattern documented"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 18a8ba6] Expand candidate diversity (30 percent different LLM choices); Ames shows clearest LLM advantage (5/7), Tox21/hERG mixed - endpoint-specific pattern documented
 1 file changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 567 bytes | 567.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   3a29e50..18a8ba6  main -> main
